# 08. OCR 모델·실행 경로 비교

07의 `gpt-5.4-mini`·Upstage 기준 결과는 수정하지 않고 참조한다. 이 노트북은 동일한 5개 gold set에 대해 OpenAI API(`gpt-5.6-terra`, `gpt-5.6-luna`, `detail=high`·`detail=original`)와 Codex CLI(Terra, Luna)를 독립 실행한다.

Codex CLI는 이미지 detail을 명시하는 CLI 옵션을 확인하지 못했으므로 `image_detail_effective=unknown`으로 기록한다. 모든 OCR 결과는 이 노트북의 `notebooks/data/08_ocr_model_runtime_comparison/` 아래에만 저장한다.

In [1]:
from __future__ import annotations
import base64, hashlib, html, json, os, re, subprocess, sys, time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import fitz
import pandas as pd
from dotenv import load_dotenv
from rapidfuzz.distance import Levenshtein

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'data' / 'raw').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError(f'프로젝트 루트를 찾지 못했습니다: {cwd}')

NOTEBOOK_ID = '08_ocr_model_runtime_comparison'
DATA_ROOT = PROJECT_ROOT / 'notebooks' / 'data' / NOTEBOOK_ID
BASELINE_ROOT = PROJECT_ROOT / 'notebooks' / 'data' / '07_goldset_vision_upstage_comparison' / '2026-07-25_live_ocr' / 'evaluation_v4'
RUN_ID = os.getenv('OCR08_RUN_ID', datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
RUN_ROOT = DATA_ROOT / 'runs' / RUN_ID
RUN_CONFIG_IDS = {item.strip() for item in os.getenv('OCR08_RUN_CONFIGS', '').split(',') if item.strip()}
RUN_FIELD_EXTRACTION = os.getenv('OCR08_RUN_FIELD_EXTRACTION', 'true').lower() == 'true'
OVERWRITE = os.getenv('OCR08_OVERWRITE', 'false').lower() == 'true'
FIELD_EXTRACTION_MODEL = os.getenv('FIELD_EXTRACTION_MODEL', 'gpt-5.4-mini')
TARGETS = [
    ('woori', 'Woori_Classic_EVERY_MILE_SKYPASS'),
    ('shinhan', 'Shinhan_Toss_Mr.Life_20251231'),
    ('samsung', 'Samsung_iD_ALL'),
    ('lotte', 'Lotte_LOCA_LIKIT_Eat'),
    ('kookmin', 'Kookmin_Friend_20210917'),
]
OCR_PROMPT = '카드 안내 PDF 페이지의 모든 텍스트를 읽기 순서대로 전사하세요. 표는 Markdown 표로 작성하고, 요약이나 추론은 하지 마세요.'
CONFIGS = [
    {'id': 'api_terra_high', 'runtime': 'openai_api', 'model': 'gpt-5.6-terra', 'image_detail_requested': 'high', 'reasoning_effort': None},
    {'id': 'api_luna_high', 'runtime': 'openai_api', 'model': 'gpt-5.6-luna', 'image_detail_requested': 'high', 'reasoning_effort': None},
    {'id': 'api_terra_original', 'runtime': 'openai_api', 'model': 'gpt-5.6-terra', 'image_detail_requested': 'original', 'reasoning_effort': None},
    {'id': 'api_luna_original', 'runtime': 'openai_api', 'model': 'gpt-5.6-luna', 'image_detail_requested': 'original', 'reasoning_effort': None},
    {'id': 'codex_cli_terra', 'runtime': 'codex_cli', 'model': 'gpt-5.6-terra', 'image_detail_requested': None, 'reasoning_effort': None},
    {'id': 'codex_cli_luna', 'runtime': 'codex_cli', 'model': 'gpt-5.6-luna', 'image_detail_requested': None, 'reasoning_effort': None},
]
CONFIG_BY_ID = {config['id']: config for config in CONFIGS}
if unknown := RUN_CONFIG_IDS - set(CONFIG_BY_ID):
    raise ValueError(f'알 수 없는 OCR08_RUN_CONFIGS 값: {sorted(unknown)}')

load_dotenv(PROJECT_ROOT / '.env')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print({'run_id': RUN_ID, 'run_configs': sorted(RUN_CONFIG_IDS), 'run_root': str(RUN_ROOT), 'field_extraction_model': FIELD_EXTRACTION_MODEL})

{'run_id': '20260807T142500Z_repeat01', 'run_configs': ['api_luna_original', 'api_terra_original'], 'run_root': '/home/sms/openclaw_file/PickCardU/notebooks/data/08_ocr_model_runtime_comparison/runs/20260807T142500Z_repeat01', 'field_extraction_model': 'gpt-5.4-mini'}


In [2]:
# 공통 입력·저장 함수. 외부 호출은 하지 않는다.
def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def read_text(path: Path) -> str:
    return Path(path).read_text(encoding='utf-8')

def write_json(path: Path, value) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')

def documents():
    rows = []
    for issuer, card_name in TARGETS:
        pdf_path = PROJECT_ROOT / 'data' / 'raw' / issuer / f'{card_name}.pdf'
        if not pdf_path.is_file():
            raise FileNotFoundError(pdf_path)
        with fitz.open(pdf_path) as pdf:
            rows.append({'issuer': issuer, 'card_name': card_name, 'pdf_path': pdf_path, 'page_count': len(pdf)})
    return rows

def render_png(pdf_path: Path, page_num: int) -> bytes:
    with fitz.open(pdf_path) as pdf:
        return pdf[page_num - 1].get_pixmap(matrix=fitz.Matrix(2, 2), alpha=False).tobytes('png')

def rendered_page_path(doc, page_num: int) -> Path:
    return RUN_ROOT / 'rendered_pages' / doc['issuer'] / doc['card_name'] / f'{page_num:03d}.png'

def ensure_rendered_page(doc, page_num: int) -> Path:
    path = rendered_page_path(doc, page_num)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(render_png(doc['pdf_path'], page_num))
    return path

def raw_path(config, doc, page_num: int) -> Path:
    suffix = '.json' if config['runtime'] == 'openai_api' else '.txt'
    return RUN_ROOT / 'raw' / config['id'] / doc['issuer'] / doc['card_name'] / f'{page_num:03d}{suffix}'

def events_path(config, doc, page_num: int) -> Path:
    return RUN_ROOT / 'events' / config['id'] / doc['issuer'] / doc['card_name'] / f'{page_num:03d}.jsonl'

def write_manifest(rows):
    payload = {
        'run_id': RUN_ID,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'ocr_prompt': OCR_PROMPT,
        'configs': CONFIGS,
        'documents': rows,
        'baseline_root': str(BASELINE_ROOT),
    }
    write_json(RUN_ROOT / 'run_manifest.json', payload)

write_manifest([{**doc, 'pdf_path': str(doc['pdf_path'])} for doc in documents()])

In [3]:
# 선택한 구성만 실행한다. 기본값 OCR08_RUN_CONFIGS=''는 안전하게 실행을 건너뛴다.
def run_openai_api(config, doc, page_num: int, image_path: Path):
    if not OPENAI_API_KEY:
        raise RuntimeError('OPENAI_API_KEY가 없습니다. 프로젝트 루트 .env를 확인하세요.')
    from openai import OpenAI
    image = base64.b64encode(image_path.read_bytes()).decode('ascii')
    response = OpenAI(api_key=OPENAI_API_KEY).responses.create(
        model=config['model'],
        input=[{'role': 'user', 'content': [
            {'type': 'input_text', 'text': OCR_PROMPT},
            {'type': 'input_image', 'image_url': f'data:image/png;base64,{image}', 'detail': config['image_detail_requested']},
        ]}],
    )
    raw = response.model_dump()
    raw['page_text'] = response.output_text.strip()
    return raw

def run_codex_cli(config, doc, page_num: int, image_path: Path, output_path: Path):
    command = [
        'codex', 'exec', '--ephemeral', '--sandbox', 'read-only',
        '--model', config['model'], '--image', str(image_path),
        '--output-last-message', str(output_path), '--json', OCR_PROMPT,
    ]
    completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=900)
    events = events_path(config, doc, page_num)
    events.parent.mkdir(parents=True, exist_ok=True)
    events.write_text(completed.stdout, encoding='utf-8')
    if completed.stderr:
        events.with_suffix('.stderr.txt').write_text(completed.stderr, encoding='utf-8')
    if completed.returncode != 0:
        raise RuntimeError(f'codex exec 종료 코드 {completed.returncode}: {completed.stderr[-1000:]}')

rows = []
for config in CONFIGS:
    if config['id'] not in RUN_CONFIG_IDS:
        continue
    for doc in documents():
        for page_num in range(1, doc['page_count'] + 1):
            output = raw_path(config, doc, page_num)
            started = time.perf_counter()
            try:
                if output.exists() and not OVERWRITE:
                    status = 'cached'
                else:
                    image_path = ensure_rendered_page(doc, page_num)
                    output.parent.mkdir(parents=True, exist_ok=True)
                    if config['runtime'] == 'openai_api':
                        write_json(output, run_openai_api(config, doc, page_num, image_path))
                    else:
                        run_codex_cli(config, doc, page_num, image_path, output)
                    status = 'created'
                rows.append({**config, 'issuer': doc['issuer'], 'card_name': doc['card_name'], 'page_num': page_num, 'status': status, 'path': str(output), 'elapsed_seconds': round(time.perf_counter() - started, 3)})
            except Exception as error:
                rows.append({**config, 'issuer': doc['issuer'], 'card_name': doc['card_name'], 'page_num': page_num, 'status': 'failed', 'path': str(output), 'error': str(error), 'elapsed_seconds': round(time.perf_counter() - started, 3)})
                pd.DataFrame(rows).to_csv(RUN_ROOT / 'ocr_manifest.csv', index=False)
                raise
ocr_manifest = pd.DataFrame(rows)
ocr_manifest.to_csv(RUN_ROOT / 'ocr_manifest.csv', index=False)
display(ocr_manifest if not ocr_manifest.empty else pd.DataFrame([{'status': 'SKIPPED', 'reason': 'OCR08_RUN_CONFIGS가 비어 있습니다.'}]))

,id,runtime,model,image_detail_requested,reasoning_effort,issuer,card_name,page_num,status,path,elapsed_seconds
0,api_terra_original,openai_api,gpt-5.6-terra,original,None,woori,Woori_Classic_EVERY_MILE_SKYPASS,1,created,/home/sms/openclaw_file/PickCardU/notebooks/da...,57.656
1,api_terra_original,openai_api,gpt-5.6-terra,original,None,woori,Woori_Classic_EVERY_MILE_SKYPASS,2,created,/home/sms/openclaw_file/PickCardU/notebooks/da...,33.556
2,api_terra_original,openai_api,gpt-5.6-terra,original,None,shinhan,Shinhan_Toss_Mr.Life_20251231,1,created,/home/sms/openclaw_file/PickCardU/notebooks/da...,62.003
3,api_terra_original,openai_api,gpt-5.6-terra,original,None,shinhan,Shinhan_Toss_Mr.Life_20251231,2,created,/home/sms/openclaw_file/PickCardU/notebooks/da...,36.999
4,api_terra_original,openai_api,gpt-5.6-terra,original,None,samsung,Samsung_iD_ALL,1,created,/home/sms/openclaw_file/PickCardU/notebooks/da...,44.166
5,api_terra_original,openai_api,gpt-5.6-terra,original,None,samsung,Samsung_iD_ALL,2,created,/home/sms/openclaw_file/PickCardU/notebooks/da...,10.235
6,api_terra_original,openai_api,gpt-5.6-terra,original,None,samsung,Samsung_iD_ALL,3,created,/home/sms/openclaw_file/PickCardU/notebooks/da...,14.537
7,api_terra_original,openai_api,gpt-5.6-terra,original,None,samsung,Samsung_iD_ALL,4,created,/home/sms/openclaw_file/PickCardU/notebooks/da...,59.625
8,api_terra_original,openai_api,gpt-5.6-terra,original,None,samsung,Samsung_iD_ALL,5,created,/home/sms/openclaw_file/PickCardU/notebooks/da...,9.624
9,api_terra_original,openai_api,gpt-5.6-terra,original,None,samsung,Samsung_iD_ALL,6,created,/home/sms/openclaw_file/PickCardU/notebooks/da...,14.070


In [4]:
# 결과물 1: 새 OCR 원문을 공통 페이지 TXT로 저장하고 gold/raw 기준 전체 OCR 지표를 계산한다.
PAGE_MARKER = re.compile(r'^\[page\s+(\d+)\]\s*$', re.I | re.M)
PANEL_MARKER = re.compile(r'^\[(?:left|center|right)_panel\]\s*$', re.I | re.M)
def page_map(text: str):
    found = list(PAGE_MARKER.finditer(text))
    return {int(match.group(1)): text[match.end():(found[i + 1].start() if i + 1 < len(found) else len(text))].strip() for i, match in enumerate(found)}
def normalize(text: str):
    text = PANEL_MARKER.sub('', html.unescape(text or '')).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^0-9a-z가-힣]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()
def prf(reference: str, candidate: str):
    a, b = reference.split(), candidate.split()
    overlap = sum((Counter(a) & Counter(b)).values())
    precision = overlap / len(b) if b else 0.0
    recall = overlap / len(a) if a else 0.0
    return precision, recall, (2 * precision * recall / (precision + recall) if precision + recall else 0.0)
def numeric_tokens(text: str):
    return Counter(re.findall(r'(?<![a-z가-힣])\d+(?:[.,]\d+)?(?:%|원|회|개월|년|일|시|마일)?', text))
def new_page_text(config, issuer, card_name, page_num):
    path = RUN_ROOT / 'raw' / config['id'] / issuer / card_name / (f'{page_num:03d}.json' if config['runtime'] == 'openai_api' else f'{page_num:03d}.txt')
    if not path.is_file():
        return None
    return json.loads(read_text(path))['page_text'] if config['runtime'] == 'openai_api' else read_text(path).strip()

engine_sources = []
for config in CONFIGS:
    engine_sources.append({'engine': config['id'], 'kind': 'new', 'config': config})
engine_sources += [
    {'engine': 'baseline_api_gpt_5_4_mini', 'kind': 'baseline', 'source_engine': 'openai'},
    {'engine': 'baseline_upstage', 'kind': 'baseline', 'source_engine': 'upstage'},
]
metrics = []
text_manifest = []
for source in engine_sources:
    for issuer, card_name in TARGETS:
        if source['kind'] == 'new':
            content = {page: new_page_text(source['config'], issuer, card_name, page) for page in range(1, 1000)}
            content = {page: text for page, text in content.items() if text is not None}
        else:
            baseline = BASELINE_ROOT / 'text' / source['source_engine'] / issuer / f'{card_name}.txt'
            content = page_map(read_text(baseline))
        if not content:
            continue
        text_path = RUN_ROOT / 'evaluation' / 'text' / source['engine'] / issuer / f'{card_name}.txt'
        text_path.parent.mkdir(parents=True, exist_ok=True)
        text_path.write_text('\n\n'.join(f'[page {page}]\n{text}' for page, text in sorted(content.items())), encoding='utf-8')
        text_manifest.append({'engine': source['engine'], 'issuer': issuer, 'card_name': card_name, 'path': str(text_path)})
        gold = page_map(read_text(PROJECT_ROOT / 'data' / 'ocr_benchmark' / 'gold' / 'raw' / issuer / f'{card_name}.txt'))
        for page_num, truth in gold.items():
            reference, candidate = normalize(truth), normalize(content.get(page_num, ''))
            precision, recall, f1 = prf(reference, candidate)
            np, nr, nf = prf(' '.join(numeric_tokens(reference).elements()), ' '.join(numeric_tokens(candidate).elements()))
            metrics.append({'engine': source['engine'], 'issuer': issuer, 'card_name': card_name, 'page_num': page_num, 'cer': Levenshtein.normalized_distance(reference, candidate), 'token_precision': precision, 'token_recall': recall, 'token_f1': f1, 'numeric_f1': nf, 'exact_match': reference == candidate})
text_manifest = pd.DataFrame(text_manifest)
full_page_metrics = pd.DataFrame(metrics)
text_manifest.to_csv(RUN_ROOT / 'evaluation' / 'text_manifest.csv', index=False)
full_page_metrics.to_csv(RUN_ROOT / 'evaluation' / 'full_page_metrics.csv', index=False)
full_summary = full_page_metrics.groupby('engine', dropna=False)[['cer', 'token_precision', 'token_recall', 'token_f1', 'numeric_f1', 'exact_match']].mean().reset_index()
display(full_summary)

,engine,cer,token_precision,token_recall,token_f1,numeric_f1,exact_match
0,api_luna_original,0.174118,0.879298,0.960230,0.892396,0.830915,0.000000
1,api_terra_original,0.139300,0.851345,0.935707,0.865549,0.798995,0.000000
2,baseline_api_gpt_5_4_mini,0.190399,0.862641,0.949205,0.882580,0.804303,0.000000
3,baseline_upstage,0.151717,0.903914,0.980367,0.919520,0.833697,0.090909


In [5]:
# 결과물 2: 동일한 고정 모델·프롬프트로 OCR TXT를 필드 JSON으로 변환하고 gold/structured와 비교한다.
def nullable(schema): return {'anyOf': [schema, {'type': 'null'}]}
def value_schema(shape):
    if isinstance(shape, dict): return nullable({'type': 'object', 'properties': {key: value_schema(value) for key, value in shape.items()}, 'required': list(shape), 'additionalProperties': False})
    if isinstance(shape, list): return nullable({'type': 'array', 'items': value_schema(shape[0]) if shape else {}})
    return nullable({'type': {'str': 'string', 'int': 'number', 'float': 'number', 'bool': 'boolean'}.get(shape, 'string')})
def shape(value):
    if isinstance(value, dict): return {key: shape(item) for key, item in value.items()}
    if isinstance(value, list): return [shape(value[0])] if value else []
    return type(value).__name__
def field_schema(gold):
    return {'field_labels': [{'id': x['id'], 'page_num': x['page_num'], 'context_terms': x.get('context_terms', []), 'value_shape': shape(x['value'])} for x in gold.get('field_labels', [])], 'numeric_labels': [{'id': x['id'], 'page_num': x['page_num'], 'context_terms': x.get('context_terms', []), 'unit': x.get('unit'), 'value_shape': {'surface_text': 'str', 'normalized_value': type(x.get('normalized_value')).__name__}} for x in gold.get('numeric_labels', [])], 'table_labels': [{'id': x['id'], 'page_num': x['page_num'], 'column_count': len(x.get('headers', []))} for x in gold.get('table_labels', [])]}
def prediction_schema(schema):
    field_props = {item['id']: value_schema(item['value_shape']) for item in schema['field_labels']}
    numeric_value = nullable({'type': 'object', 'properties': {'surface_text': nullable({'type': 'string'}), 'normalized_value': nullable({'anyOf': [{'type': 'string'}, {'type': 'number'}]})}, 'required': ['surface_text', 'normalized_value'], 'additionalProperties': False})
    numeric_props = {item['id']: numeric_value for item in schema['numeric_labels']}
    cell = nullable({'anyOf': [{'type': 'string'}, {'type': 'number'}]})
    table_value = nullable({'type': 'object', 'properties': {'headers': {'type': 'array', 'items': cell}, 'rows': {'type': 'array', 'items': {'type': 'array', 'items': cell}}}, 'required': ['headers', 'rows'], 'additionalProperties': False})
    table_props = {item['id']: table_value for item in schema['table_labels']}
    return {'type': 'object', 'properties': {'field_labels': {'type': 'object', 'properties': field_props, 'required': list(field_props), 'additionalProperties': False}, 'numeric_labels': {'type': 'object', 'properties': numeric_props, 'required': list(numeric_props), 'additionalProperties': False}, 'table_labels': {'type': 'object', 'properties': table_props, 'required': list(table_props), 'additionalProperties': False}}, 'required': ['field_labels', 'numeric_labels', 'table_labels'], 'additionalProperties': False}
def canonical(value): return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
def extract_fields(ocr_text, schema):
    if not OPENAI_API_KEY: raise RuntimeError('FIELD_EXTRACTION_MODEL 호출에는 OPENAI_API_KEY가 필요합니다.')
    from openai import OpenAI
    prompt = 'OCR 텍스트만 근거로 스키마의 각 필드를 추출하세요. OCR에 없는 값은 null로 두고 추정·보정하지 마세요. 원문 표기는 유지하고 normalized_value만 숫자 또는 날짜 같은 정규값으로 쓰세요. 표는 headers와 rows를 반환하세요. JSON 구조와 모든 field_id는 제공된 JSON Schema를 반드시 따르세요.' + '\n\nVALUE-LESS SCHEMA:\n' + json.dumps(schema, ensure_ascii=False) + '\n\nOCR TEXT:\n' + ocr_text
    response = OpenAI(api_key=OPENAI_API_KEY).responses.create(model=FIELD_EXTRACTION_MODEL, input=prompt, text={'format': {'type': 'json_schema', 'name': 'ocr_field_prediction', 'strict': True, 'schema': prediction_schema(schema)}}, store=False)
    return json.loads(response.output_text)

field_rows = []
for source in engine_sources:
    for issuer, card_name in TARGETS:
        text_path = RUN_ROOT / 'evaluation' / 'text' / source['engine'] / issuer / f'{card_name}.txt'
        if not text_path.is_file():
            continue
        output = RUN_ROOT / 'evaluation' / 'structured' / source['engine'] / issuer / f'{card_name}.json'
        gold = json.loads(read_text(PROJECT_ROOT / 'data' / 'ocr_benchmark' / 'gold' / 'structured' / issuer / f'{card_name}.json'))
        if source['kind'] == 'baseline':
            baseline = BASELINE_ROOT / 'structured' / source['source_engine'] / issuer / f'{card_name}.json'
            prediction = json.loads(read_text(baseline))
        elif output.exists() and not OVERWRITE:
            prediction = json.loads(read_text(output))
        elif RUN_FIELD_EXTRACTION:
            prediction = extract_fields(read_text(text_path), field_schema(gold))
            write_json(output, prediction)
        else:
            continue
        for kind in ('field_labels', 'numeric_labels', 'table_labels'):
            for label in gold.get(kind, []):
                field_id = label['id']
                if kind == 'field_labels': truth = label['value']
                elif kind == 'numeric_labels': truth = {'surface_text': label['surface_text'], 'normalized_value': label['normalized_value']}
                else: truth = {'headers': label['headers'], 'rows': label['rows']}
                candidate = prediction.get(kind, {}).get(field_id)
                field_rows.append({'engine': source['engine'], 'issuer': issuer, 'card_name': card_name, 'kind': kind, 'field_id': field_id, 'critical': bool(label.get('critical', False)), 'exact_match': canonical(truth) == canonical(candidate)})
field_metrics = pd.DataFrame(field_rows)
field_metrics.to_csv(RUN_ROOT / 'evaluation' / 'field_exact_metrics.csv', index=False)
field_summary = field_metrics.groupby('engine', dropna=False).agg(values=('field_id', 'count'), exact_matches=('exact_match', 'sum'), exact_match_rate=('exact_match', 'mean')).reset_index() if not field_metrics.empty else pd.DataFrame()
display(field_summary)

,engine,values,exact_matches,exact_match_rate
0,api_luna_original,117,51,0.435897
1,api_terra_original,95,33,0.347368
2,baseline_api_gpt_5_4_mini,117,46,0.393162
3,baseline_upstage,117,44,0.376068


In [6]:
# 결과물 3: 실행·평가 요약. 실패한 페이지와 모델별 결과 경로를 함께 기록한다.
summary = {
    'notebook_id': NOTEBOOK_ID,
    'run_id': RUN_ID,
    'run_root': str(RUN_ROOT),
    'baseline_root': str(BASELINE_ROOT),
    'requested_configs': sorted(RUN_CONFIG_IDS),
    'ocr_prompt': OCR_PROMPT,
    'field_extraction_model': FIELD_EXTRACTION_MODEL,
    'codex_cli_image_detail': 'unknown: codex exec 도움말에 high/original 명시 옵션이 없음',
    'files': {
        'run_manifest': 'run_manifest.json',
        'ocr_manifest': 'ocr_manifest.csv',
        'full_page_metrics': 'evaluation/full_page_metrics.csv',
        'field_exact_metrics': 'evaluation/field_exact_metrics.csv',
    },
}
write_json(RUN_ROOT / 'summary.json', summary)
display(pd.DataFrame([summary]))

,notebook_id,run_id,run_root,baseline_root,requested_configs,ocr_prompt,field_extraction_model,codex_cli_image_detail,files
0,08_ocr_model_runtime_comparison,20260807T142500Z_repeat01,/home/sms/openclaw_file/PickCardU/notebooks/da...,/home/sms/openclaw_file/PickCardU/notebooks/da...,"[api_luna_original, api_terra_original]",카드 안내 PDF 페이지의 모든 텍스트를 읽기 순서대로 전사하세요. 표는 Markd...,gpt-5.4-mini,unknown: codex exec 도움말에 high/original 명시 옵션이 없음,"{'run_manifest': 'run_manifest.json', 'ocr_man..."
